# GridMind GR — Exploratory Data Analysis

A concise review of the ADMIE system load and RES datasets stored in PostgreSQL.

The raw tables keep periods 1–25. Period 25 is inspected separately because the forecasting pipeline uses a standard 24-hour daily grid.


In [ ]:
import sys

import matplotlib.pyplot as plt
import pandas as pd

sys.path.append("../src")

from gridmind.data.database import get_engine
from sqlalchemy import text


In [ ]:
load_query = """
    SELECT date, period, net_load_mwh, crete_flow_mwh
    FROM system_load
    ORDER BY date, period;
"""

res_query = """
    SELECT date, period, res_mwh
    FROM res_production
    ORDER BY date, period;
"""

with get_engine().connect() as connection:
    system_load = pd.read_sql_query(text(load_query), connection)

with get_engine().connect() as connection:
    res = pd.read_sql_query(text(res_query), connection)

system_load["date"] = pd.to_datetime(system_load["date"])
res["date"] = pd.to_datetime(res["date"])

print(
    f"System Load: {system_load['date'].min().date()} "
    f"-> {system_load['date'].max().date()} | {len(system_load):,} rows"
)
print(
    f"RES:         {res['date'].min().date()} "
    f"-> {res['date'].max().date()} | {len(res):,} rows"
)


## Data quality


In [ ]:
quality = pd.DataFrame(
    {
        "rows": [len(system_load), len(res)],
        "missing_values": [
            int(system_load.isna().sum().sum()),
            int(res.isna().sum().sum()),
        ],
        "duplicate_date_period": [
            int(system_load.duplicated(["date", "period"]).sum()),
            int(res.duplicated(["date", "period"]).sum()),
        ],
    },
    index=["system_load", "res_production"],
)

quality


In [ ]:
periods_per_day = (
    system_load.groupby("date")["period"]
    .count()
    .value_counts()
    .sort_index()
)

periods_per_day


## Load patterns


In [ ]:
daily_load = system_load[system_load["period"] <= 24].groupby("date")["net_load_mwh"].mean()

plt.figure(figsize=(14, 5))
plt.plot(daily_load.index, daily_load.values)
plt.title("Average Daily System Load")
plt.xlabel("Date")
plt.ylabel("Load (MWh)")
plt.tight_layout()
plt.show()


In [ ]:
hourly_load = (
    system_load[system_load["period"] <= 24]
    .groupby("period")["net_load_mwh"]
    .mean()
)

plt.figure(figsize=(10, 5))
plt.plot(hourly_load.index - 1, hourly_load.values, marker="o")
plt.title("Average System Load by Hour")
plt.xlabel("Hour")
plt.ylabel("Load (MWh)")
plt.xticks(range(24))
plt.tight_layout()
plt.show()


In [ ]:
weekday_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday",
]

weekday_load = (
    system_load[system_load["period"] <= 24].assign(day_of_week=system_load["date"].dt.day_name())
    .groupby("day_of_week")["net_load_mwh"]
    .mean()
    .reindex(weekday_order)
)

weekday_load


## RES and load


In [ ]:
energy = system_load.merge(
    res,
    on=["date", "period"],
    how="inner",
)

energy_24h = energy[energy["period"] <= 24].copy()

print(
    "Correlation between load and RES:",
    round(energy_24h["net_load_mwh"].corr(energy_24h["res_mwh"]), 3),
)

energy_24h[
    ["net_load_mwh", "res_mwh"]
].describe()


## Period 25


In [ ]:
period_25 = system_load[system_load["period"] == 25].copy()
period_25_nonzero = period_25[period_25["net_load_mwh"] != 0]

print(f"Period 25 rows: {len(period_25):,}")
print(f"Non-zero period 25 rows: {len(period_25_nonzero)}")

period_25_nonzero[
    ["date", "period", "net_load_mwh", "crete_flow_mwh"]
]


### EDA conclusion

The PostgreSQL data is structurally consistent and suitable for the v1 forecasting pipeline.  
For modeling, GridMind uses periods 1–24 to maintain a uniform hourly grid. Period 25 remains preserved in the raw database for DST and data-quality inspection.
